# Notebook 80: V8.9.1 Forced Signed-Band Removal Stress Test

Quick-and-dirty diagnostic on the 151-page HITL roster. It runs two deliberately aggressive treatments against the corrected V8.8 parent:

- **Arm A, OUTER_0_TO_2:** force Telea reconstruction on seed boundary Ring 0 and outward Rings +1 and +2.
- **Arm B, SIGNED_N2_TO_P2:** force Telea reconstruction on inward Rings -2 and -1, seed boundary Ring 0, and outward Rings +1 and +2.

This is expected to fail visual review. It is not an equivalence, release or promotion notebook. It performs only basic containment, outside-authority immutability, output generation and exact replay on one smoke-test page. No clean truth is used.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scikit-learn joblib torch torchvision pyarrow matplotlib


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, importlib.util, sys
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
OPERATOR_SOURCE=ROOT/'v8_9_release'/'v8_9_1_operator.py'
P_PRINT_BUNDLE=ROOT/'observed_print_full_development_freeze_v8_2f'/'20261117_120424_v8_2f_full_development_p_print_freeze'/'p_print_provider'
REMOVE_BUNDLE=ROOT/'m5_remove_full_development_freeze_v8_8f'/'20261117_110512_notebook52f_v8_8_full_development_remove_freeze'/'full_development_remove'
NB72_ROOT=ROOT/'notebook72_frozen_v8_8_annotated_only_operational_trial'
EXPECTED_PAGES=151
RUN=ROOT/'v8_9_1_stress_test'/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook80')
for p in [RUN,RUN/'pages',RUN/'tables',RUN/'visual',RUN/'audit',RUN/'upload']:
    p.mkdir(parents=True,exist_ok=False)

def require(x,m):
    if not bool(x): raise RuntimeError(m)

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def latest(root,required):
    hits=[p for p in Path(root).rglob('*') if p.is_dir() and all((p/r).exists() for r in required)]
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

require(OPERATOR_SOURCE.exists(),f'Missing {OPERATOR_SOURCE}')
spec=importlib.util.spec_from_file_location('v8_9_1_operator',OPERATOR_SOURCE)
mod=importlib.util.module_from_spec(spec)
sys.modules['v8_9_1_operator']=mod
spec.loader.exec_module(mod)

op_outer=mod.load_v891_operator(P_PRINT_BUNDLE,REMOVE_BUNDLE,mode='OUTER_0_TO_2')
op_signed=mod.load_v891_operator(P_PRINT_BUNDLE,REMOVE_BUNDLE,mode='SIGNED_N2_TO_P2')
require(type(op_outer).__name__=='V891Operator','OUTER operator load failure.')
require(type(op_signed).__name__=='V891Operator','SIGNED operator load failure.')

NB72=latest(NB72_ROOT,['inputs/runtime_roster.csv'])
require(NB72 is not None,'Complete Notebook 72 run not found.')
roster=pd.read_csv(NB72/'inputs/runtime_roster.csv')
required={'page','observed_path','seed_mask_path'}
require(required<=set(roster.columns),f'Runtime roster missing {sorted(required-set(roster.columns))}')
require(len(roster)==EXPECTED_PAGES,f'Expected 151 HITL pages, found {len(roster)}')
print('Run:',RUN)
print('Pages:',len(roster))


## One-page smoke test

This is intentionally minimal. It checks that both arms are confined to dilation-2 authority, preserve the V8.8 parent outside the selected bands, and replay exactly on one page.


In [ ]:
r=roster.iloc[0]
page=str(r.page)
obs=cv2.imread(str(r.observed_path),cv2.IMREAD_GRAYSCALE)
seed=cv2.imread(str(r.seed_mask_path),cv2.IMREAD_GRAYSCALE)
require(obs is not None and seed is not None and obs.shape==seed.shape,f'{page}: input failure')

a1=op_outer.run_page(obs,seed,page=page)
a2=op_outer.run_page(obs,seed,page=page)
b1=op_signed.run_page(obs,seed,page=page)
b2=op_signed.run_page(obs,seed,page=page)

checks={
    'outer_exact_replay':np.array_equal(a1.corrected,a2.corrected),
    'signed_exact_replay':np.array_equal(b1.corrected,b2.corrected),
    'outer_outside_forced_parent_exact':np.array_equal(a1.corrected[~a1.forced_remove_mask],a1.parent_v88_corrected[~a1.forced_remove_mask]),
    'signed_outside_forced_parent_exact':np.array_equal(b1.corrected[~b1.forced_remove_mask],b1.parent_v88_corrected[~b1.forced_remove_mask]),
    'outer_inside_authority':bool(np.all(a1.forced_remove_mask<=a1.authoritative_mask)),
    'signed_inside_authority':bool(np.all(b1.forced_remove_mask<=b1.authoritative_mask)),
}
require(all(checks.values()),f'Smoke test failed: {checks}')
print(json.dumps(checks,indent=2))
print('Outer forced pixels:',int(a1.forced_remove_mask.sum()),'changed:',int(a1.v891_delta_mask.sum()))
print('Signed forced pixels:',int(b1.forced_remove_mask.sum()),'changed:',int(b1.v891_delta_mask.sum()))


## Generate both stress-test arms across all 151 pages

The operator is run separately for each arm. Outputs are deliberately verbose enough for visual review, but no truth scoring or release-equivalence suite is performed.


In [ ]:
rows=[]
for _,r in roster.iterrows():
    page=str(r.page)
    obs=cv2.imread(str(r.observed_path),cv2.IMREAD_GRAYSCALE)
    seed=cv2.imread(str(r.seed_mask_path),cv2.IMREAD_GRAYSCALE)
    require(obs is not None and seed is not None and obs.shape==seed.shape,f'{page}: input failure')

    outer=op_outer.run_page(obs,seed,page=page)
    signed=op_signed.run_page(obs,seed,page=page)

    d=RUN/'pages'/page
    d.mkdir(parents=True,exist_ok=False)
    cv2.imwrite(str(d/'observed.png'),obs)
    cv2.imwrite(str(d/'authoritative_seed_mask.png'),seed)
    cv2.imwrite(str(d/'v8_8_parent.png'),outer.parent_v88_corrected)
    cv2.imwrite(str(d/'v8_9_1_outer_0_to_2.png'),outer.corrected)
    cv2.imwrite(str(d/'v8_9_1_signed_n2_to_p2.png'),signed.corrected)
    cv2.imwrite(str(d/'outer_0_to_2_mask.png'),outer.forced_remove_mask.astype(np.uint8)*255)
    cv2.imwrite(str(d/'signed_n2_to_p2_mask.png'),signed.forced_remove_mask.astype(np.uint8)*255)
    cv2.imwrite(str(d/'outer_delta.png'),outer.v891_delta_mask.astype(np.uint8)*255)
    cv2.imwrite(str(d/'signed_delta.png'),signed.v891_delta_mask.astype(np.uint8)*255)
    np.save(str(d/'signed_ring_map.npy'),signed.signed_ring_map)
    (d/'page_audit.json').write_text(json.dumps({'outer_0_to_2':outer.v891_audit,'signed_n2_to_p2':signed.v891_audit},indent=2))

    rows.append({
        'page':page,
        'ring_minus2_pixels':int(signed.ring_minus2_mask.sum()),
        'ring_minus1_pixels':int(signed.ring_minus1_mask.sum()),
        'ring0_pixels':int(signed.ring0_mask.sum()),
        'ring1_pixels':int(signed.ring1_mask.sum()),
        'ring2_pixels':int(signed.ring2_mask.sum()),
        'outer_forced_pixels':int(outer.forced_remove_mask.sum()),
        'outer_changed_vs_v88':int(outer.v891_delta_mask.sum()),
        'signed_forced_pixels':int(signed.forced_remove_mask.sum()),
        'signed_changed_vs_v88':int(signed.v891_delta_mask.sum()),
        'outer_vs_signed_different_pixels':int(np.sum(outer.corrected!=signed.corrected)),
        'outer_outside_forced_parent_mismatch':int(np.sum(outer.corrected[~outer.forced_remove_mask]!=outer.parent_v88_corrected[~outer.forced_remove_mask])),
        'signed_outside_forced_parent_mismatch':int(np.sum(signed.corrected[~signed.forced_remove_mask]!=signed.parent_v88_corrected[~signed.forced_remove_mask])),
    })
    print(page,'outer',int(outer.v891_delta_mask.sum()),'signed',int(signed.v891_delta_mask.sum()))

results=pd.DataFrame(rows)
results.to_csv(RUN/'tables'/'151_page_v8_9_1_stress_test.csv',index=False)
require(len(results)==151,'Incomplete 151-page output.')
require((results.outer_outside_forced_parent_mismatch==0).all(),'Outer arm changed non-selected parent pixels.')
require((results.signed_outside_forced_parent_mismatch==0).all(),'Signed arm changed non-selected parent pixels.')
print('Generated:',len(results),'pages x 2 arms')


## Contact sheets ranked by the extreme arm’s changed-pixel count

In [ ]:
ranked=results.sort_values('signed_changed_vs_v88',ascending=False).reset_index(drop=True)
ranked.to_csv(RUN/'tables'/'151_page_v8_9_1_stress_test_ranked.csv',index=False)

CONTACT_SIZE=8
contact_paths=[]
for start in range(0,len(ranked),CONTACT_SIZE):
    q=ranked.iloc[start:start+CONTACT_SIZE]
    fig,axes=plt.subplots(len(q),4,figsize=(16,max(5,5*len(q))),squeeze=False)
    for row_index,(_,row) in enumerate(q.iterrows()):
        d=RUN/'pages'/str(row.page)
        paths=[d/'observed.png',d/'v8_8_parent.png',d/'v8_9_1_outer_0_to_2.png',d/'v8_9_1_signed_n2_to_p2.png']
        titles=['Observed','V8.8 parent','V8.9.1 rings 0..+2','V8.9.1 rings -2..+2']
        for col,(path,title) in enumerate(zip(paths,titles)):
            im=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)
            axes[row_index,col].imshow(im,cmap='gray',vmin=0,vmax=255)
            axes[row_index,col].set_title(f'{row.page}\n{title}' if col==0 else title,fontsize=8)
            axes[row_index,col].axis('off')
    plt.tight_layout()
    path=RUN/'visual'/f'contact_sheet_{start//CONTACT_SIZE+1:02d}.png'
    plt.savefig(path,dpi=110,bbox_inches='tight')
    plt.close(fig)
    contact_paths.append(path)
print('Contact sheets:',len(contact_paths))


## Diagnostic summary and upload package

In [ ]:
summary={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'version':'8.9.1-quick-dirty-stress-test',
    'status':'DIAGNOSTIC_ONLY_EXPECTED_TO_FAIL',
    'pages':int(len(results)),
    'arms':{
        'OUTER_0_TO_2':'forced Telea on seed boundary Ring 0 plus outward Rings +1 and +2',
        'SIGNED_N2_TO_P2':'forced Telea on inward Rings -2/-1, boundary Ring 0 and outward Rings +1/+2',
    },
    'outer_total_forced_pixels':int(results.outer_forced_pixels.sum()),
    'outer_total_changed_vs_v88':int(results.outer_changed_vs_v88.sum()),
    'signed_total_forced_pixels':int(results.signed_forced_pixels.sum()),
    'signed_total_changed_vs_v88':int(results.signed_changed_vs_v88.sum()),
    'outside_selected_parent_mismatches':int(results.outer_outside_forced_parent_mismatch.sum()+results.signed_outside_forced_parent_mismatch.sum()),
    'uses_clean_truth':False,
    'promotion_authority':False,
    'operator_sha256':sha256_file(OPERATOR_SOURCE),
}
(RUN/'diagnostic_summary.json').write_text(json.dumps(summary,indent=2))

readme='''NOTEBOOK 80 - V8.9.1 FORCED SIGNED-BAND REMOVAL STRESS TEST

Diagnostic only. Expected to fail visual review.

Arm A: forced Telea on Ring 0 and outward Rings +1/+2.
Arm B: forced Telea on signed Rings -2 through +2.

Start with diagnostic_summary.json, the ranked CSV and contact sheets.
'''
(RUN/'upload'/'README.txt').write_text(readme)
for p in [RUN/'diagnostic_summary.json',RUN/'tables'/'151_page_v8_9_1_stress_test.csv',RUN/'tables'/'151_page_v8_9_1_stress_test_ranked.csv',*contact_paths]:
    shutil.copy2(p,RUN/'upload'/p.name)
shutil.copy2(OPERATOR_SOURCE,RUN/'upload'/'v8_9_1_operator.py')
shutil.make_archive(str(RUN/'notebook80_upload_bundle'),'zip',RUN/'upload')
print(json.dumps(summary,indent=2))
print('COMPLETE:',RUN)
